Khai báo thư viện

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import haversine_distances
from math import radians

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from statsmodels.api import WLS
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [6]:
folder_data = r"/content/gdrive/MyDrive/BI/Đồ án cuối kỳ/Ý tưởng Thanh Tâm/Final Round - Dataset"

df_buyer = pd.read_excel(folder_data + r"/Buyer.xlsx", sheet_name= "Buyer")
df_sellers = pd.read_excel(folder_data + r"/Seller.xlsx", sheet_name= "Seller")

df_geolocation = pd.read_excel(folder_data + r"/Geolocation.xlsx", sheet_name= "Geolocation")

df_orders = pd.read_excel(folder_data + r"/Order.xlsx", sheet_name= "Order")
df_order_items = pd.read_excel(folder_data + r"/Order Item.xlsx", sheet_name= "Order item")
df_products = pd.read_excel(folder_data + r"/Product.xlsx", sheet_name= "Product")

In [8]:
# 2. Gộp tọa độ cho Người mua (Buyer)
master_df = pd.merge(df_orders, df_buyer, on='buyer_id', how='left')
master_df = master_df.merge(df_geolocation, left_on='buyer_zip_code_prefix',
                            right_on='geolocation_zip_code_prefix', how='left')
master_df.rename(columns={'geolocation_lat': 'buyer_lat', 'geolocation_lng': 'buyer_lng'}, inplace=True)
master_df.drop(columns=['geolocation_zip_code_prefix'], inplace=True)

# Gộp với Order Items (Lưu ý: 1 đơn có thể có nhiều items)
master_df = pd.merge(master_df, df_order_items, on='order_id', how='left')

# Gộp với Products để lấy thông tin vật lý sản phẩm
master_df = pd.merge(master_df, df_products, on='product_id', how='left')

# Gộp với Sellers để biết thông tin người bán
master_df = pd.merge(master_df, df_sellers, on='seller_id', how='left')
master_df = master_df.merge(df_geolocation, left_on='seller_zip_code_prefix',
                            right_on='geolocation_zip_code_prefix', how='left')
master_df.rename(columns={'geolocation_lat': 'seller_lat', 'geolocation_lng': 'seller_lng'}, inplace=True)
master_df.drop(columns=['geolocation_zip_code_prefix'], inplace=True)

# 5. Xử lý định dạng thời gian cho toàn bộ bảng
time_columns = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_buyer_date',
    'order_estimated_delivery_date', 'shipping_limit_date',
]

for col in time_columns:
    if col in master_df.columns:
        master_df[col] = pd.to_datetime(master_df[col])

# Kiểm tra kết quả
print(f"Tổng số cột: {len(master_df.columns)}")
print(f"Tổng số dòng: {len(master_df)}")
master_df.head()

Tổng số cột: 37
Tổng số dòng: 113405


,order_id,buyer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_buyer_date,order_estimated_delivery_date,buyer_unique_id,buyer_zip_code_prefix,...,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,seller_lat,seller_lng,geolocation_city_y,geolocation_state_y
0,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2023-09-04 21:15:19,2023-10-07 13:18:03,2023-10-18 13:14:51,NaT,2023-10-20,b7d76e111c89f7ebf14761390f0f7d17,69309.0,...,32.0,6.0,28.0,37580.0,monte siao,MG,-22.428805,-46.569550,monte siao,MG
1,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2023-09-04 21:15:19,2023-10-07 13:18:03,2023-10-18 13:14:51,NaT,2023-10-20,b7d76e111c89f7ebf14761390f0f7d17,69309.0,...,32.0,6.0,28.0,37580.0,monte siao,MG,-22.428805,-46.569550,monte siao,MG
2,e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,canceled,2023-09-05 00:15:34,2023-10-07 13:17:15,NaT,NaT,2023-10-28,4854e9b3feff728c13ee5fc7d1547e92,99025.0,...,25.0,2.0,25.0,81050.0,curitiba,PR,-25.495038,-49.299601,curitiba,PR
3,809a282bbd5dbcabb6f2f724fca862ec,622e13439d6b5a0b486c435618b2679e,canceled,2023-09-13 15:24:19,2023-10-07 13:16:46,NaT,NaT,2023-09-30,009b0127b727ab0ba422f6d9604487c7,12244.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2023-09-15 12:16:38,2023-09-15 12:16:38,2023-11-07 17:11:53,2023-11-09 07:47:38,2023-10-04,830d5b7aaa3b6f1e9ad63703bec97d23,14600.0,...,16.0,16.0,16.0,81810.0,curitiba,PR,-25.507144,-49.272075,curitiba,PR


Tính khoảng cách người mua và người bán

In [9]:
def calculate_distance(row):
    # Nếu thiếu tọa độ thì trả về NaN
    if pd.isna(row['buyer_lat']) or pd.isna(row['seller_lat']):
        return np.nan

    buyer = [radians(row['buyer_lat']), radians(row['buyer_lng'])]
    seller = [radians(row['seller_lat']), radians(row['seller_lng'])]

    # Tính khoảng cách theo KM (bán kính Trái Đất ~ 6371km)
    result = haversine_distances([buyer, seller]) * 6371
    return result[0][1]

master_df['distance_km'] = master_df.apply(calculate_distance, axis=1)

In [10]:
master_df[['buyer_lat','buyer_lng','seller_lat','seller_lng','distance_km']].head()

,buyer_lat,buyer_lng,seller_lat,seller_lng,distance_km
0,2.812997,-60.695264,-22.428805,-46.569550,3198.535766
1,2.812997,-60.695264,-22.428805,-46.569550,3198.535766
2,-28.261098,-52.407671,-25.495038,-49.299601,435.420648
3,-23.204901,-45.950860,NaN,NaN,NaN
4,-20.581177,-47.858931,-25.507144,-49.272075,566.488927


xử lý dữ liệu thời gian

In [15]:
# Danh sách các cột chứa dữ liệu thời gian trong bảng của bạn
datetime_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_buyer_date',
    'order_estimated_delivery_date', 'shipping_limit_date'
]

# Chuyển đổi tất cả về định dạng datetime
for col in datetime_cols:
    if col in master_df.columns:
        master_df[col] = pd.to_datetime(master_df[col])

In [16]:
# Tạo cột Tháng
master_df['order_purchase_month'] = master_df['order_purchase_timestamp'].dt.month

# Tạo thêm các cột hữu ích cho dự đoán thời gian giao hàng
master_df['order_purchase_day_of_week'] = master_df['order_purchase_timestamp'].dt.dayofweek # 0=Thứ 2, 6=Chủ nhật
master_df['is_weekend'] = master_df['order_purchase_day_of_week'].isin([5, 6]).astype(int) # 1 nếu là cuối tuần, 0 nếu không

master_df['order_purchase_hour'] = master_df['order_purchase_timestamp'].dt.hour # Giờ trong ngày (0-23)
master_df['is_working'] = master_df['order_purchase_hour'].isin([9, 10, 11, 12, 13, 14, 15, 16, 17]).astype(int) # 1 nếu là giờ làm việc, 0 nếu không

In [18]:
# 1. Thời gian giao hàng thực tế (Target variable) - tính bằng ngày
# Chú ý: Chỉ tính được cho các đơn đã giao (delivered)
master_df['actual_delivery_days'] = (master_df['order_delivered_buyer_date'] - master_df['order_purchase_timestamp']).dt.total_seconds() / 86400

# 2. Thời gian xử lý của người bán (Từ lúc đặt -> Giao cho đơn vị vận chuyển)
master_df['seller_process_days'] = ((master_df['order_delivered_carrier_date'] - master_df['order_purchase_timestamp']).dt.total_seconds() / 86400).round(0)

# 3. Thời gian giao hàng dự kiến của hệ thống
master_df['estimated_delivery_days'] = (master_df['order_estimated_delivery_date'] - master_df['order_purchase_timestamp']).dt.total_seconds() / 86400

# 4. Thời gian giới hạn giao hàng (shipping limit)
master_df['shipping_limit_days'] = (master_df['shipping_limit_date'] - master_df['order_purchase_timestamp']).dt.total_seconds() / 86400
master_df['is_shipping_limited'] = (master_df['actual_delivery_days'] < (master_df['shipping_limit_days'])).astype(int)

In [19]:
master_df[['is_shipping_limited','shipping_limit_days','actual_delivery_days']].head()

,is_shipping_limited,shipping_limit_days,actual_delivery_days
0,0,51.881944,NaN
1,0,51.881944,NaN
2,0,14.000000,NaN
3,0,NaN,NaN
4,0,4.454803,54.813194


In [20]:
import holidays
from datetime import timedelta

# 1. Khởi tạo lịch lễ của Brazil (BR)
br_holidays = holidays.BR()

def check_near_holiday(row, window_days=7):
    # Lấy ngày đặt hàng
    purchase_date = row['order_purchase_timestamp'].date()

    # Kiểm tra trong vòng 7 ngày tới có ngày lễ nào không
    for i in range(1, window_days + 1):
        future_date = purchase_date + timedelta(days=i)
        if future_date in br_holidays:
            return 1
    return 0

# 3. Áp dụng vào DataFrame của bạn
# Lưu ý: Đảm bảo các cột thời gian đã được chuyển sang dạng datetime
master_df['has_holiday'] = master_df.apply(check_near_holiday, axis=1)

Tạo biến xem rằng cùng bang/ thành phố có ảnh hưởng

In [21]:
master_df['same_state'] = (master_df['buyer_state'] == master_df['seller_state']).astype(int)
master_df['same_city'] = (master_df['buyer_city'] == master_df['seller_city']).astype(int)

xử lý dữ kiện về kích thước đơn hàng --> thể tích

In [22]:
# Tính thể tích sản phẩm (cm3)
master_df['product_volume_cm3'] = (
    master_df['product_length_cm'] * master_df['product_height_cm'] * master_df['product_width_cm']
)

kiểm tra xem đơn hàng có bao nhiêu người bán

In [23]:
# Đếm số người bán duy nhất cho mỗi đơn hàng
seller_counts = master_df.groupby('order_id')['seller_id'].nunique().sort_values(ascending=False)

print("Top 5 đơn hàng có nhiều người bán nhất:")
print(seller_counts.head())

Top 5 đơn hàng có nhiều người bán nhất:
order_id
1c11d0f4353b31ac3417fbfa5f0f2a8a    5
cf5c8d9f52807cb2d2f0a0ff54c478da    5
8c2b13adf3f377c8f2b06b04321b0925    4
1d23106803c48c391366ff224513fb7f    4
91be51c856a90d7efe86cf9d082d6ae3    4
Name: seller_id, dtype: int64

Phân bổ số lượng người bán trên mỗi đơn hàng:
seller_id
1    97.951158
2     1.226087
0     0.763413
3     0.054314
4     0.003017
5     0.002012
Name: proportion, dtype: float64


kiểm tra các missing value

In [27]:
import pandas as pd

def check_missing_data(df):
    # Tính số lượng giá trị thiếu trên mỗi cột
    missing_count = df.isnull().sum()

    # Tính tỉ lệ phần trăm (%) thiếu
    missing_percent = (df.isnull().sum() / len(df)) * 100

    # Gộp thành một DataFrame để dễ quan sát
    missing_report = pd.DataFrame({
        'Số lượng thiếu': missing_count,
        'Tỉ lệ (%)': missing_percent
    })

    # Sắp xếp theo tỉ lệ giảm dần và chỉ hiện các cột có thiếu dữ liệu
    missing_report = missing_report[missing_report['Số lượng thiếu'] > 0].sort_values(by='Tỉ lệ (%)', ascending=False)

    return missing_report

# Chạy thử hàm
missing_info = check_missing_data(master_df)
print(missing_info.head(10))  # In ra top 10 cột có tỉ lệ thiếu cao nhất
cols_to_drop_80 = missing_info[missing_info['Tỉ lệ (%)'] > 80].index.tolist()

# In ra danh sách để kiểm tra
print("Các cột có tỉ lệ thiếu > 80% cần xóa:")
print(cols_to_drop_80)

                            Số lượng thiếu  Tỉ lệ (%)
distance_km                           3467   3.057184
actual_delivery_days                  3210   2.830563
order_delivered_buyer_date            3210   2.830563
seller_lat                            3028   2.670076
geolocation_city_y                    3028   2.670076
geolocation_state_y                   3028   2.670076
seller_lng                            3028   2.670076
seller_state                          2775   2.446982
seller_zip_code_prefix                2775   2.446982
seller_city                           2775   2.446982
Các cột có tỉ lệ thiếu > 80% cần xóa:
[]


In [28]:
# Danh sách các cột datetime ban đầu
cols_to_drop = [
    #id
    'buyer_id', 'buyer_unique_id', 'order_item_id',
    #dữ liệu thời gian
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_buyer_date',
    'order_estimated_delivery_date',  'shipping_limit_date','order_purchase_day_of_week','order_purchase_hour',
    #các cột liên quan đến địa lý
    'seller_city', 'buyer_city','buyer_lat', 'buyer_lng', 'seller_lat', 'seller_lng',
    #mã vùng
    'buyer_zip_code_prefix', 'seller_zip_code_prefix',
    #kích thước sản phẩm
    'product_length_cm', 'product_height_cm', 'product_width_cm',
    #nội dung và hình ảnh sản phẩm
    'product_name_lenght', 'product_description_lenght', 'product_photos_qty',
    #trạng thái giao hàng
    'order_status'
]

# Chỉ xóa những cột tồn tại trong DataFrame để tránh lỗi
existing_cols_to_drop = [c for c in cols_to_drop if c in master_df.columns] + [c for c in cols_to_drop_80 if c not in cols_to_drop]
col_not_in_master = [c for c in cols_to_drop if c not in master_df.columns]
print(f"Các cột: {col_not_in_master} không có trong master_df và sẽ không bị xóa.")

# Tạo một bản sao mới cho việc training (giữ lại master_df cũ để đối soát) và chỉ giữ lại các đơn đã giao và đang chờ giao để đảm bảo target variable có ý nghĩa
df_train = master_df[master_df['order_status'] =='delivered'].drop(columns=existing_cols_to_drop).dropna()
df_test = master_df[master_df['order_status'] =='shipped'].drop(columns=existing_cols_to_drop).dropna()

Các cột: [] không có trong master_df và sẽ không bị xóa.


encode dữ liệu

In [29]:
df_train.select_dtypes(include=['object']).info()

<class 'pandas.core.frame.DataFrame'>
Index: 107395 entries, 4 to 113397
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   order_id               107395 non-null  object
 1   buyer_state            107395 non-null  object
 2   geolocation_city_x     107395 non-null  object
 3   geolocation_state_x    107395 non-null  object
 4   product_id             107395 non-null  object
 5   seller_id              107395 non-null  object
 6   product_category_name  107395 non-null  object
 7   seller_state           107395 non-null  object
 8   geolocation_city_y     107395 non-null  object
 9   geolocation_state_y    107395 non-null  object
dtypes: object(10)
memory usage: 9.0+ MB


In [32]:
from category_encoders import TargetEncoder

# 1. Phân nhóm các cột
high_card_cols = ['product_category_name','buyer_state','seller_state']

# 2. Xử lý Target Encoding (Thay tên bằng giá trị trung bình của biến mục tiêu)
# Điều này giúp mô hình hiểu được "mức độ trễ" trung bình của từng thành phố
te = TargetEncoder(cols=high_card_cols)
df_train_encode = te.fit_transform(df_train, df_train['actual_delivery_days'])

# 3. Kiểm tra kết quả
print("Danh sách các cột sau khi Encode:")
print(df_train_encode.columns.tolist())
print(f"\nKích thước DataFrame mới: {df_train_encode.shape}")


Danh sách các cột sau khi Encode:
['order_id', 'buyer_state', 'geolocation_city_x', 'geolocation_state_x', 'product_id', 'seller_id', 'sale_value', 'freight_value', 'product_category_name', 'product_weight_g', 'seller_state', 'geolocation_city_y', 'geolocation_state_y', 'distance_km', 'order_purchase_month', 'is_weekend', 'is_working', 'actual_delivery_days', 'seller_process_days', 'estimated_delivery_days', 'shipping_limit_days', 'is_shipping_limited', 'has_holiday', 'same_state', 'same_city', 'product_volume_cm3', 'num_sellers']

Kích thước DataFrame mới: (107395, 27)


In [54]:
df_train_encode_group_id = df_train_encode.groupby('order_id').agg({
    'actual_delivery_days': 'first',  # Target variable
    'same_state': 'min', # đơn hàng có giao khác ban không?
    'same_city': 'min', # đơn hàng có giao khác thành phố không?
    'sale_value': 'sum',  # Tổng giá trị đơn hàng
    'freight_value': 'sum',  # Tổng giá trị vận chuyển
    'product_weight_g': 'sum',  # Tổng trọng lượng sản phẩm trong đơn hàng
    'product_volume_cm3': 'sum',  # Tổng thể tích sản phẩm trong đơn hàng
    'order_purchase_month': 'first',  # Lấy tháng diễn ra đơn hàng
    'seller_process_days': 'max', # thời gian xử lý của người bán lâu nhất trong đơn hàng
    'estimated_delivery_days': 'first', # thời gian giao hàng
    'shipping_limit_days': 'min', # thời gian giới hạn giao hàng ngắn nhất trong đơn hàng
    'distance_km': 'max',
    'has_holiday': 'max', # kiểm tra xem có sản phẩm nào trong đơn hàng bị ảnh hưởng bởi ngày lễ không
    'is_weekend': 'max', # kiểm tra xem có sản phẩm nào trong đơn hàng được đặt vào cuối tuần không
    'is_working': 'min', # kiểm tra xem có sản phẩm nào trong đơn hàng được thanh toán ngoài giờ làm việc không
    'seller_id': 'nunique', # số lượng người bán trong đơn hàng
    'product_id': 'count' # số lượng sản phẩm trong đơn hàng
    }).copy()
df_train_encode_group_id.rename(columns={'seller_id': 'num_sellers',
                                         'product_id': 'num_products'
                                         }, inplace=True)

In [55]:
def calculate_weight(row):
    delay = row['actual_delivery_days'] - row['shipping_limit_days']
    if delay > 0:
        return 1 + delay # Trễ càng nhiều, trọng số càng nặng
    else:
        return 1 # Đúng hạn thì coi như bình thường

df_train_encode_group_id['weights'] = df_train_encode_group_id.apply(calculate_weight, axis=1)

In [56]:
df_train_encode_group_id.info()

<class 'pandas.core.frame.DataFrame'>
Index: 94353 entries, 00010242fe8c5a6d1ba2dd792cb16214 to fffe41c64501cc87c801fd61db3f6244
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   actual_delivery_days     94353 non-null  float64
 1   same_state               94353 non-null  int64  
 2   same_city                94353 non-null  int64  
 3   sale_value               94353 non-null  float64
 4   freight_value            94353 non-null  float64
 5   product_weight_g         94353 non-null  float64
 6   product_volume_cm3       94353 non-null  float64
 7   order_purchase_month     94353 non-null  int32  
 8   seller_process_days      94353 non-null  float64
 9   estimated_delivery_days  94353 non-null  float64
 10  shipping_limit_days      94353 non-null  float64
 11  distance_km              94353 non-null  float64
 12  has_holiday              94353 non-null  int64  
 13  is_weekend             

In [57]:
def remove_outliers_iqr_list(df, columns_list):

    df_clean = df.copy()
    initial_count = len(df_clean)

    for col in columns_list:
        # Tính toán Q1, Q3 và IQR cho từng cột
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Lọc dữ liệu trực tiếp trên df_clean
        before_count = len(df_clean)
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
        after_count = len(df_clean)

        print(f"[*] Đã xử lý cột: {col:<25} | Loại bỏ: {before_count - after_count:>6} dòng")

    total_removed = initial_count - len(df_clean)
    print("-" * 60)
    print(f"Tổng cộng đã loại bỏ: {total_removed} dòng ({total_removed/initial_count*100:.2f}%)")

    return df_clean

cols_to_fix = ['actual_delivery_days','shipping_limit_days','estimated_delivery_days','seller_process_days','product_weight_g','product_volume_cm3', 'distance_km', 'freight_value','sale_value']

# Chạy hàm
df_train_clean = remove_outliers_iqr_list(df_train_encode_group_id, cols_to_fix)

[*] Đã xử lý cột: actual_delivery_days      | Loại bỏ:   4945 dòng
[*] Đã xử lý cột: shipping_limit_days       | Loại bỏ:   6788 dòng
[*] Đã xử lý cột: estimated_delivery_days   | Loại bỏ:   1646 dòng
[*] Đã xử lý cột: seller_process_days       | Loại bỏ:   2106 dòng
[*] Đã xử lý cột: product_weight_g          | Loại bỏ:  10857 dòng
[*] Đã xử lý cột: product_volume_cm3        | Loại bỏ:   4327 dòng
[*] Đã xử lý cột: distance_km               | Loại bỏ:   4268 dòng
[*] Đã xử lý cột: freight_value             | Loại bỏ:   4681 dòng
[*] Đã xử lý cột: sale_value                | Loại bỏ:   3465 dòng
------------------------------------------------------------
Tổng cộng đã loại bỏ: 43083 dòng (45.66%)


Scaler

In [58]:
from sklearn.preprocessing import StandardScaler

# 1. Khởi tạo
scaler = StandardScaler()

Huấn luyện mô hình

In [59]:
df_train_encode_group_id.shape

(94353, 18)

In [60]:
X = df_train_encode_group_id.drop(columns=['actual_delivery_days', 'weights']).copy()
X = scaler.fit_transform(X)
X = pd.DataFrame(X, columns=df_train_encode_group_id.drop(columns=['actual_delivery_days', 'weights']).columns)

y = df_train_encode_group_id['actual_delivery_days'].copy()

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [61]:
# 1. Danh sách các mô hình cần thử nghiệm
models = [
    ("Linear Regression", LinearRegression()),
    ("Ridge (L2)", Ridge(alpha=1.0)),
    ("Lasso (L1)", Lasso(alpha=0.1)),
    ("ElasticNet", ElasticNet(alpha=0.1, l1_ratio=0.5)),
    ("Random Forest", RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)),
    ("XGBoost", XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)),
    ("LightGBM", LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
]

# 2.1 Dictionary để lưu trữ kết quả Feature Importance của từng mô hình
all_importances = {}

# 2.2 Tạo một list để lưu kết quả
results = []

print(f"{'Model':<20} | {'MAE':<10} | {'RMSE':<10} | {'R2 Score':<10}")
print("-" * 55)
for name, model in models:
    # Huấn luyện mô hình
    model.fit(X_train, Y_train)
    # Dự đoán
    Y_pred = model.predict(X_test)

    # Tính toán các chỉ số
    mae = mean_absolute_error(Y_test, Y_pred)
    rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
    r2 = r2_score(Y_test, Y_pred)

    # Lưu kết quả
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })
    print(f"{name:<20} | {mae:<10.2f} | {rmse:<10.2} | {r2:<10.2f}")


    # --- TRÍCH XUẤT ĐỘ QUAN TRỌNG ---
    if hasattr(model, 'coef_'):
        values = model.coef_
    elif hasattr(model, 'feature_importances_'):
        values = model.feature_importances_
    else:
        continue

    # Tạo DataFrame cho mô hình hiện tại
    imp_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Value': values,
        'Abs_Value': pd.Series(values).abs()
    }).sort_values(by='Abs_Value', ascending=False).reset_index(drop=True)

    # Lưu vào Dictionary tổng
    all_importances[name] = imp_df

Model                | MAE        | RMSE       | R2 Score  
-------------------------------------------------------
Linear Regression    | 4.58       | 7.5        | 0.34      
Ridge (L2)           | 4.58       | 7.5        | 0.34      
Lasso (L1)           | 4.58       | 7.5        | 0.35      
ElasticNet           | 4.58       | 7.5        | 0.35      
Random Forest        | 4.25       | 7.1        | 0.41      
XGBoost              | 4.16       | 6.9        | 0.44      
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008888 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1886
[LightGBM] [Info] Number of data points in the train set: 75482, number of used features: 16
[LightGBM] [Info] Start training from score 12.499122
LightGBM             | 4.16       | 6.9        | 0.44      


=> có chất lượng mô hình tương đồng giữa các mô hình phi tuyến và các mô hình tuyến tính.

=> Nhưng mô hình phi tuyến có sự chênh lệch rõ ràng và thấy được bộ dữ liệu hoạt động tốt ở mô hình phi tuyến

In [62]:
df_importances = pd.DataFrame({
    'Random Forest': all_importances['Random Forest']['Feature'],
    'XGBoost': all_importances['XGBoost']['Feature'],
    "LightGBM": all_importances['LightGBM']['Feature']})

In [63]:
df_importances

,Random Forest,XGBoost,LightGBM
0,seller_process_days,same_state,order_purchase_month
1,same_state,seller_process_days,distance_km
2,distance_km,distance_km,seller_process_days
3,order_purchase_month,order_purchase_month,estimated_delivery_days
4,estimated_delivery_days,estimated_delivery_days,shipping_limit_days
5,shipping_limit_days,has_holiday,freight_value
6,product_volume_cm3,num_sellers,product_volume_cm3
7,freight_value,product_volume_cm3,sale_value
8,sale_value,num_products,product_weight_g
9,product_weight_g,freight_value,num_products


In [65]:
selected_top_features = 5
imp_features = (df_importances['Random Forest'].head(selected_top_features).to_list()+df_importances['XGBoost'].head(selected_top_features).to_list()+df_importances['LightGBM'].head(selected_top_features).to_list())
imp_features = list(set(imp_features))
print(f"Top {selected_top_features} features từ mỗi mô hình (Random Forest, XGBoost, Light GBM) có {len(imp_features)} biến:")
print(imp_features)

Top 5 features từ mỗi mô hình (Random Forest, XGBoost, Light GBM) có 6 biến:
['seller_process_days', 'same_state', 'shipping_limit_days', 'distance_km', 'estimated_delivery_days', 'order_purchase_month']


In [66]:
Best_col = imp_features + ['weights', 'actual_delivery_days']  # Thêm cột target vào danh sách các cột quan trọng
df_train_new =df_train_clean[Best_col].copy()

In [67]:
df_train_new


,seller_process_days,same_state,shipping_limit_days,distance_km,estimated_delivery_days,order_purchase_month,weights,actual_delivery_days
order_id,,,,,,,,
00010242fe8c5a6d1ba2dd792cb16214,6.0,0,6.032326,301.005664,15.625671,9,2.582095,7.614421
000229ec398224ef6ca0657da4fc703e,2.0,1,4.010405,312.495046,21.393391,1,4.938032,7.948437
00024acbcdf0a6daa1e931b038114c75,2.0,1,7.006748,301.951753,11.582928,8,1.000000,6.147269
00048cc3ae777c65dbb7d2a0634bc1ea,2.0,0,7.258947,161.597746,21.095440,5,1.000000,6.668067
00054e8431b9d7675808bcb819fb4a32,2.0,1,4.011609,484.118172,24.504306,12,5.411887,8.423495
...,...,...,...,...,...,...,...,...
fff90cdcb3b2e6cfb397d05d562fd3fe,4.0,1,6.047002,15.932451,13.622373,11,1.000000,4.722662
fffb2ef8874127f75b52b643880fd7e0,4.0,0,9.563704,876.529078,27.349433,3,8.491725,17.055428
fffce4705a9662cd70adb13d4a31832d,3.0,0,7.004502,338.827218,17.286157,10,1.000000,4.801690


In [68]:
from sklearn.model_selection import RandomizedSearchCV

Test trên mô hình chưa gán trọng số

In [70]:
X = df_train_new.drop(columns=['actual_delivery_days', 'weights']).copy()
X = scaler.fit_transform(X)
X = pd.DataFrame(X, columns=df_train_new.drop(columns=['actual_delivery_days', 'weights']).columns)

y = df_train_new['actual_delivery_days'].copy()

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [71]:
# 1. Danh sách các mô hình cần thử nghiệm
models = [
    ("Random Forest", RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)),
    ("XGBoost", XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)),
    ("LightGBM", LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
]

# 2.1 Dictionary để lưu trữ kết quả Feature Importance của từng mô hình
all_importances = {}

# 2.2 Tạo một list để lưu kết quả
results = []

print(f"{'Model':<20} | {'MAE':<10} | {'RMSE':<10} | {'R2 Score':<10}")
print("-" * 55)
for name, model in models:
    # Huấn luyện mô hình
    model.fit(X_train, Y_train)
    # Dự đoán
    Y_pred = model.predict(X_test)

    # Tính toán các chỉ số
    mae = mean_absolute_error(Y_test, Y_pred)
    rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
    r2 = r2_score(Y_test, Y_pred)

    # Lưu kết quả
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })
    print(f"{name:<20} | {mae:<10.2f} | {rmse:<10.2} | {r2:<10.2f}")


    # --- TRÍCH XUẤT ĐỘ QUAN TRỌNG ---
    if hasattr(model, 'coef_'):
        values = model.coef_
    elif hasattr(model, 'feature_importances_'):
        values = model.feature_importances_
    else:
        continue

    # Tạo DataFrame cho mô hình hiện tại
    imp_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Value': values,
        'Abs_Value': pd.Series(values).abs()
    }).sort_values(by='Abs_Value', ascending=False).reset_index(drop=True)

    # Lưu vào Dictionary tổng
    all_importances[name] = imp_df

Model                | MAE        | RMSE       | R2 Score  
-------------------------------------------------------
Random Forest        | 3.00       | 4.1        | 0.46      
XGBoost              | 2.95       | 4.1        | 0.47      
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005156 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 791
[LightGBM] [Info] Number of data points in the train set: 41016, number of used features: 6
[LightGBM] [Info] Start training from score 9.747743
LightGBM             | 2.96       | 4.1        | 0.47      


Mô hình có trọng số: tập trung bắt các đơn hàng có nguy cơ trễ cao

In [72]:
X = df_train_new.drop(columns=['actual_delivery_days', 'weights']).copy()
y = df_train_new['actual_delivery_days']

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_weights=df_train_new['weights'].loc[Y_train.index].values

In [ ]:
models = [
    ("Random Forest", RandomForestRegressor(random_state=42), {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None],
        'bootstrap': [True, False]
    }),

    ("XGBoost", XGBRegressor(random_state=42), {
        'n_estimators': [100, 200, 300, 500],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 6, 9, 12],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'gamma': [0, 0.1, 0.2, 0.3],
        'reg_alpha': [0, 0.01, 0.1, 1],
        'reg_lambda': [0.5, 1, 1.5, 2]
    }),

    ("LightGBM", LGBMRegressor(random_state=42, verbose=-1), {
        'n_estimators': [100, 200, 300, 500, 1000],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'num_leaves': [20, 31, 50, 100, 150],
        'max_depth': [-1, 10, 20, 30],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'reg_alpha': [0, 0.01, 0.1, 1],
        'reg_lambda': [0.5, 1, 1.5, 2]
    })
]

# 2.2 Tạo một list để lưu kết quả
results = []

print(f"{'Model':<20} | {'MAE':<10} | {'RMSE':<10} | {'R2 Score':<10}")
print("-" * 55)
for name, model, param_distributions in models:
    # Huấn luyện mô hình
    random_search = RandomizedSearchCV(estimator=model,
                           param_distributions=param_distributions,
                           n_iter=50,
                           scoring='r2',
                           cv=5,
                           n_jobs=-1,
                           verbose=2,
                           random_state=42)

    random_search.fit(X_train, Y_train, sample_weight=train_weights)

    model = random_search.best_estimator_

    # Dự đoán
    Y_pred = model.predict(X_test)

    # Tính toán các chỉ số
    mae = mean_absolute_error(Y_test, Y_pred)
    rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
    r2 = r2_score(Y_test, Y_pred)

    # Lưu kết quả
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'best params': random_search.best_params_
    })
    print(f"{name:<20} | {mae:<10.2f} | {rmse:<10.2} | {r2:<10.2f}")

Model                | MAE        | RMSE       | R2 Score  
-------------------------------------------------------
Fitting 5 folds for each of 50 candidates, totalling 250 fits
